# GMM Classification with Test-Set Fit on SNGP Embeddings

Load saved SNGP embeddings, fit label-assigned full-covariance GMMs on the **CIFAR-10 test set** and **training set**, and compare the fitted distributions.

Reported metrics use clear split-specific naming:
- GMM posterior accuracy on test set
- GMM posterior accuracy on training set
- Train-fit vs test-fit class mean distances
- Train-fit vs test-fit covariance spread comparison



In [1]:
from pathlib import Path
import os
import sys

import numpy as np

cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / "src").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError(f"Could not find repo root from {cwd}")

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

print("repo root:", REPO_ROOT)


repo root: /w/20252/wjcai/uq/manygp


In [2]:
EMBEDDING_PATH = REPO_ROOT / "notebooks" / "gmm" / "sngp_embeddings_cifar10_sngp_epoch175_acc0.9323.npz"

if not EMBEDDING_PATH.exists():
    candidates = sorted((REPO_ROOT / "notebooks" / "gmm").glob("sngp_embeddings_*.npz"))
    if not candidates:
        raise FileNotFoundError("No sngp_embeddings_*.npz files found in notebooks/gmm")
    EMBEDDING_PATH = candidates[-1]

data = np.load(EMBEDDING_PATH, allow_pickle=True)

train_embeddings = data["train_embeddings"].astype(np.float64)
train_labels = data["train_labels"].astype(np.int64)
test_embeddings = data["test_embeddings"].astype(np.float64)
test_labels = data["test_labels"].astype(np.int64)
classes = data["classes"]
checkpoint_id = str(data["checkpoint_id"]) if "checkpoint_id" in data else EMBEDDING_PATH.stem.removeprefix("sngp_embeddings_")

print("embedding file:", EMBEDDING_PATH)
print("checkpoint id:", checkpoint_id)
print("train embeddings:", train_embeddings.shape)
print("test embeddings:", test_embeddings.shape)
print("classes:", classes.tolist())


embedding file: /w/20252/wjcai/uq/manygp/notebooks/gmm/sngp_embeddings_cifar10_sngp_epoch175_acc0.9323.npz
checkpoint id: cifar10_sngp_epoch175_acc0.9323
train embeddings: (50000, 128)
test embeddings: (10000, 128)
classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


Fit label-assigned GMMs with labels fixed on each split: `z = y_train` for the train-fit GMM and `z = y_test` for the test-fit GMM. The class prior `p(z)` is the empirical class frequency in the fitted split, and `p(x | z)` is a full-covariance Gaussian estimated from that split's embeddings for that class.


In [3]:
def fit_label_assigned_gmm(x, y, num_classes, covariance_reg=1e-3):
    dim = x.shape[1]
    means = np.zeros((num_classes, dim), dtype=np.float64)
    covariances = np.zeros((num_classes, dim, dim), dtype=np.float64)
    priors = np.zeros(num_classes, dtype=np.float64)

    global_variance = np.var(x, axis=0).mean()
    reg = covariance_reg * max(global_variance, 1e-12)

    for cls in range(num_classes):
        class_x = x[y == cls]
        if len(class_x) == 0:
            raise ValueError(f"class {cls} has no samples")

        priors[cls] = len(class_x) / len(x)
        means[cls] = class_x.mean(axis=0)
        centered = class_x - means[cls]
        cov = centered.T @ centered / max(len(class_x) - 1, 1)
        covariances[cls] = cov + reg * np.eye(dim)

    return priors, means, covariances, reg


def logsumexp(a, axis=None, keepdims=False):
    a_max = np.max(a, axis=axis, keepdims=True)
    out = a_max + np.log(np.sum(np.exp(a - a_max), axis=axis, keepdims=True))
    if not keepdims:
        out = np.squeeze(out, axis=axis)
    return out


def gaussian_logpdf_by_class(x, means, covariances):
    num_classes, dim = means.shape
    logpdf = np.empty((x.shape[0], num_classes), dtype=np.float64)
    log_2pi = dim * np.log(2.0 * np.pi)

    for cls in range(num_classes):
        chol = np.linalg.cholesky(covariances[cls])
        diff = (x - means[cls]).T
        solved = np.linalg.solve(chol, diff)
        mahalanobis = np.sum(solved * solved, axis=0)
        logdet = 2.0 * np.sum(np.log(np.diag(chol)))
        logpdf[:, cls] = -0.5 * (log_2pi + logdet + mahalanobis)

    return logpdf


def gmm_probabilities(x, priors, means, covariances):
    log_likelihood = gaussian_logpdf_by_class(x, means, covariances)
    log_joint = np.log(priors)[None, :] + log_likelihood
    log_px = logsumexp(log_joint, axis=1)
    posterior = np.exp(log_joint - log_px[:, None])

    return {
        "log_likelihood": log_likelihood,
        "log_joint": log_joint,
        "log_px": log_px,
        "posterior": posterior,
    }


num_classes = len(classes)
train_fit_priors, train_fit_means, train_fit_covariances, train_fit_covariance_reg = fit_label_assigned_gmm(
    train_embeddings,
    train_labels,
    num_classes=num_classes,
    covariance_reg=1e-3,
)

test_fit_priors, test_fit_means, test_fit_covariances, test_fit_covariance_reg = fit_label_assigned_gmm(
    test_embeddings,
    test_labels,
    num_classes=num_classes,
    covariance_reg=1e-3,
)

test_fit_test_gmm = gmm_probabilities(
    test_embeddings,
    test_fit_priors,
    test_fit_means,
    test_fit_covariances,
)
test_fit_train_gmm = gmm_probabilities(
    train_embeddings,
    test_fit_priors,
    test_fit_means,
    test_fit_covariances,
)

print("train-fit class priors:", np.round(train_fit_priors, 4))
print("train-fit covariance diagonal regularizer:", train_fit_covariance_reg)
print("test-fit class priors:", np.round(test_fit_priors, 4))
print("test-fit covariance diagonal regularizer:", test_fit_covariance_reg)
print("test-fit log p(x) range on test set:", (test_fit_test_gmm["log_px"].min(), test_fit_test_gmm["log_px"].max()))
print("test-fit log p(x) range on training set:", (test_fit_train_gmm["log_px"].min(), test_fit_train_gmm["log_px"].max()))



train-fit class priors: [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
train-fit covariance diagonal regularizer: 0.0006467281316112034
test-fit class priors: [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
test-fit covariance diagonal regularizer: 0.0005947019625565877
test-fit log p(x) range on test set: (232.99158937280626, 301.36118408017074)
test-fit log p(x) range on training set: (179.30214739602076, 301.3363075195674)


Compare the train-fit and test-fit GMM parameters class by class. Mean closeness is reported as raw Euclidean distance and as a distance normalized by the test-fit class standard deviation scale. Covariance spread is compared with trace, mean variance per embedding dimension, average eigenvalue, largest eigenvalue, and log determinant.


In [4]:
def covariance_spread_summary(covariances):
    dim = covariances.shape[-1]
    traces = np.trace(covariances, axis1=1, axis2=2)
    eigvals = np.linalg.eigvalsh(covariances)
    sign, logdets = np.linalg.slogdet(covariances)
    if not np.all(sign > 0):
        raise ValueError("all covariance matrices must be positive definite")

    return {
        "trace": traces,
        "mean_variance": traces / dim,
        "mean_eigenvalue": eigvals.mean(axis=1),
        "largest_eigenvalue": eigvals[:, -1],
        "logdet": logdets,
    }


def print_metric_table(headers, rows, formats):
    widths = [len(header) for header in headers]
    formatted_rows = []
    for row in rows:
        formatted = []
        for value, fmt in zip(row, formats):
            formatted_value = format(value, fmt) if fmt else str(value)
            formatted.append(formatted_value)
        formatted_rows.append(formatted)
        widths = [max(width, len(value)) for width, value in zip(widths, formatted)]

    print("  ".join(header.ljust(width) for header, width in zip(headers, widths)))
    print("  ".join("-" * width for width in widths))
    for row in formatted_rows:
        print("  ".join(value.rjust(width) for value, width in zip(row, widths)))


mean_diffs = train_fit_means - test_fit_means
mean_l2_distances = np.linalg.norm(mean_diffs, axis=1)
test_fit_class_scales = np.sqrt(np.trace(test_fit_covariances, axis1=1, axis2=2) / test_fit_covariances.shape[-1])
train_fit_class_scales = np.sqrt(np.trace(train_fit_covariances, axis1=1, axis2=2) / train_fit_covariances.shape[-1])
pooled_class_scales = np.sqrt(0.5 * (test_fit_class_scales**2 + train_fit_class_scales**2))
mean_l2_distances_in_test_std = mean_l2_distances / test_fit_class_scales
mean_l2_distances_in_pooled_std = mean_l2_distances / pooled_class_scales

mean_rows = []
for cls, class_name in enumerate(classes):
    mean_rows.append((
        int(cls),
        str(class_name),
        mean_l2_distances[cls],
        mean_l2_distances_in_test_std[cls],
        mean_l2_distances_in_pooled_std[cls],
    ))

print("Mean comparison: train-fit minus test-fit")
print_metric_table(
    ["class", "name", "l2", "l2/test_std", "l2/pooled_std"],
    mean_rows,
    ["d", "", ".6f", ".6f", ".6f"],
)
print()
print("Mean distance summary:")
print(f"  average l2 distance: {mean_l2_distances.mean():.6f}")
print(f"  max l2 distance: {mean_l2_distances.max():.6f} ({classes[mean_l2_distances.argmax()]})")
print(f"  average l2/test_std: {mean_l2_distances_in_test_std.mean():.6f}")
print(f"  max l2/test_std: {mean_l2_distances_in_test_std.max():.6f} ({classes[mean_l2_distances_in_test_std.argmax()]})")
print(f"  average l2/pooled_std: {mean_l2_distances_in_pooled_std.mean():.6f}")
print(f"  max l2/pooled_std: {mean_l2_distances_in_pooled_std.max():.6f} ({classes[mean_l2_distances_in_pooled_std.argmax()]})")

train_spread = covariance_spread_summary(train_fit_covariances)
test_spread = covariance_spread_summary(test_fit_covariances)

spread_rows = []
for cls, class_name in enumerate(classes):
    spread_rows.append((
        int(cls),
        str(class_name),
        train_spread["trace"][cls],
        test_spread["trace"][cls],
        train_spread["trace"][cls] / test_spread["trace"][cls],
        train_spread["mean_variance"][cls],
        test_spread["mean_variance"][cls],
        train_spread["largest_eigenvalue"][cls] / test_spread["largest_eigenvalue"][cls],
        train_spread["logdet"][cls] - test_spread["logdet"][cls],
    ))

print()
print("Covariance spread comparison: ratio < 1 means train-fit covariance is smaller than test-fit")
print_metric_table(
    ["class", "name", "train_trace", "test_trace", "trace_ratio", "train_mean_var", "test_mean_var", "maxeig_ratio", "logdet_diff"],
    spread_rows,
    ["d", "", ".6f", ".6f", ".6f", ".6f", ".6f", ".6f", ".6f"],
)
print()
print("Covariance spread summary:")
print(f"  classes with smaller train-fit trace: {np.sum(train_spread['trace'] < test_spread['trace'])}/{num_classes}")
print(f"  average trace ratio: {(train_spread['trace'] / test_spread['trace']).mean():.6f}")
print(f"  average mean-variance ratio: {(train_spread['mean_variance'] / test_spread['mean_variance']).mean():.6f}")
print(f"  average largest-eigenvalue ratio: {(train_spread['largest_eigenvalue'] / test_spread['largest_eigenvalue']).mean():.6f}")
print(f"  average logdet difference, train minus test: {(train_spread['logdet'] - test_spread['logdet']).mean():.6f}")


Mean comparison: train-fit minus test-fit
class  name        l2        l2/test_std  l2/pooled_std
-----  ----------  --------  -----------  -------------
    0    airplane  0.862359     2.414966       2.683836
    1  automobile  0.425150     1.341167       1.418604
    2        bird  1.282940     3.496436       3.961756
    3         cat  1.698141     4.398683       5.143280
    4        deer  0.928205     2.587657       2.856990
    5         dog  1.591502     4.030353       4.542093
    6        frog  0.696159     1.916868       2.082120
    7       horse  0.702021     1.992623       2.182035
    8        ship  0.606695     1.823099       1.964430
    9       truck  0.483425     1.433864       1.549763

Mean distance summary:
  average l2 distance: 0.927660
  max l2 distance: 1.698141 (cat)
  average l2/test_std: 2.543572
  max l2/test_std: 4.398683 (cat)
  average l2/pooled_std: 2.838491
  max l2/pooled_std: 5.143280 (cat)

Covariance spread comparison: ratio < 1 means train-fit cov

For each split, compute the class-posterior prediction `argmax p_gmm(y | x)` and report the corresponding accuracy.



In [5]:
test_set_posterior_pred = test_fit_test_gmm["posterior"].argmax(axis=1)
training_set_posterior_pred = test_fit_train_gmm["posterior"].argmax(axis=1)

gmm_posterior_accuracy_on_test_set = np.mean(test_set_posterior_pred == test_labels)
gmm_posterior_accuracy_on_training_set = np.mean(training_set_posterior_pred == train_labels)

print(f"GMM posterior accuracy on test set: {gmm_posterior_accuracy_on_test_set * 100:.2f}%")
print(f"GMM posterior accuracy on training set: {gmm_posterior_accuracy_on_training_set * 100:.2f}%")



GMM posterior accuracy on test set: 93.32%
GMM posterior accuracy on training set: 100.00%


In [6]:
# result_path = REPO_ROOT / "notebooks" / "gmm" / f"gmm_test_fit_results_{checkpoint_id}.npz"
# np.savez_compressed(
#     result_path,
#     train_fit_priors=train_fit_priors,
#     train_fit_means=train_fit_means,
#     train_fit_covariances=train_fit_covariances,
#     train_fit_covariance_reg=train_fit_covariance_reg,
#     test_fit_priors=test_fit_priors,
#     test_fit_means=test_fit_means,
#     test_fit_covariances=test_fit_covariances,
#     test_fit_covariance_reg=test_fit_covariance_reg,
#     test_fit_log_px_on_test_set=test_fit_test_gmm["log_px"],
#     test_fit_log_px_on_training_set=test_fit_train_gmm["log_px"],
#     test_fit_posterior_probs_on_test_set=test_fit_test_gmm["posterior"],
#     test_fit_posterior_probs_on_training_set=test_fit_train_gmm["posterior"],
#     test_set_posterior_pred=test_set_posterior_pred,
#     training_set_posterior_pred=training_set_posterior_pred,
#     gmm_posterior_accuracy_on_test_set=gmm_posterior_accuracy_on_test_set,
#     gmm_posterior_accuracy_on_training_set=gmm_posterior_accuracy_on_training_set,
#     mean_l2_distances=mean_l2_distances,
#     mean_l2_distances_in_test_std=mean_l2_distances_in_test_std,
#     mean_l2_distances_in_pooled_std=mean_l2_distances_in_pooled_std,
#     train_covariance_trace=train_spread["trace"],
#     test_covariance_trace=test_spread["trace"],
#     train_covariance_mean_variance=train_spread["mean_variance"],
#     test_covariance_mean_variance=test_spread["mean_variance"],
#     train_covariance_largest_eigenvalue=train_spread["largest_eigenvalue"],
#     test_covariance_largest_eigenvalue=test_spread["largest_eigenvalue"],
#     train_covariance_logdet=train_spread["logdet"],
#     test_covariance_logdet=test_spread["logdet"],
#     train_labels=train_labels,
#     test_labels=test_labels,
#     classes=classes,
#     checkpoint_id=checkpoint_id,
#     embedding_path=str(EMBEDDING_PATH),
# )
# print("saved:", result_path)

